# 03 — Mapping Layer (Logical Targets → CSS Selectors)

This notebook loads the IR produced by Notebook 01 and maps logical element
names (e.g., `new_todo_input`, `todo_item`) to real CSS selectors for the
TodoMVC application.

Pipeline:
NL → IR → Mapped IR → Code → Execution → Analysis


In [22]:
from pathlib import Path
import json


## 1. Load IR from Disk


In [23]:
ir_path = Path("../sample-data/ir_examples/home_page_ir.json")

with open(ir_path, "r") as f:
    ir = json.load(f)

ir


{'test_name': 'render_app_test',
 'description': 'Test Name: Home Page Test\nGoal: Verify the home page loads and shows expected content.\nURL: https://ai-test-automation-demo.onrender.com/\n\nSteps:\n1. Navigate to the home page.\n2. Verify the page loads successfully.\n3. Verify the heading "Hello from your Flask Demo App!" is visible.\n4. Verify the "Login" link is visible.\n5. Click the "Login" link.\n6. Verify the browser navigates to /login.\n\nExpected Result:\nThe home page loads and displays the correct heading and login link.\n',
 'steps': [{'action': 'navigate',
   'target': 'url',
   'value': 'https://ai-test-automation-demo.onrender.com/'},
  {'action': 'navigate',
   'target': 'url',
   'value': 'https://ai-test-automation-demo.onrender.com/login'},
  {'action': 'click', 'target': 'login_button'}],
 'metadata': {'priority': 'high'}}

## 2. Define the TodoMVC Mapping Model

This maps logical names → real CSS selectors.


In [24]:
APP_MODEL = {
    "username_field": "input[name='username']",
    "password_field": "input[name='password']",
    "login_button": "button[type='submit']",

    "dashboard_heading": "h1",   # "Welcome to the Dashboard!"
    "error_message": "p.error",  # if you added a class; if not, update below

    "back_to_home_link": "a[href='/']",
    "logout_link": "a[href='/logout']"
}


## 3. Apply Mapping to IR Steps


In [25]:
def map_ir_targets(ir, model):
    mapped_steps = []

    for step in ir["steps"]:
        mapped_step = step.copy()

        target = step.get("target")
        if target in model:
            mapped_step["selector"] = model[target]
        else:
            mapped_step["selector"] = None  # unmapped target

        mapped_steps.append(mapped_step)

    mapped_ir = ir.copy()
    mapped_ir["steps"] = mapped_steps
    return mapped_ir


## 4. Generate Mapped IR


In [26]:
mapped_ir = map_ir_targets(ir, APP_MODEL)
mapped_ir


{'test_name': 'render_app_test',
 'description': 'Test Name: Home Page Test\nGoal: Verify the home page loads and shows expected content.\nURL: https://ai-test-automation-demo.onrender.com/\n\nSteps:\n1. Navigate to the home page.\n2. Verify the page loads successfully.\n3. Verify the heading "Hello from your Flask Demo App!" is visible.\n4. Verify the "Login" link is visible.\n5. Click the "Login" link.\n6. Verify the browser navigates to /login.\n\nExpected Result:\nThe home page loads and displays the correct heading and login link.\n',
 'steps': [{'action': 'navigate',
   'target': 'url',
   'value': 'https://ai-test-automation-demo.onrender.com/',
   'selector': None},
  {'action': 'navigate',
   'target': 'url',
   'value': 'https://ai-test-automation-demo.onrender.com/login',
   'selector': None},
  {'action': 'click',
   'target': 'login_button',
   'selector': "button[type='submit']"}],
 'metadata': {'priority': 'high'}}

## 5. Save Mapped IR for Downstream Notebooks


In [27]:
output_path = Path("../sample-data/ir_examples/home_page_mapped_ir.json")
output_path.parent.mkdir(parents=True, exist_ok=True)

with open(output_path, "w") as f:
    json.dump(mapped_ir, f, indent=4)

output_path


PosixPath('../sample-data/ir_examples/home_page_mapped_ir.json')